# XGBoost Model

This notebook runs the training data with the XGBoost model to see how well it performs. 

In [16]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline
from sklearn.base import clone
import xgboost as xgb

from sklearn.model_selection import cross_val_score, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import make_scorer, classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve

import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
path = ['./Data/X_train.csv', './Data/y_train.csv']
X_train_temp, y_train_temp = [pd.read_csv(f, index_col=0) for f in path]


In [4]:
top_1000_genes_temp = pd.read_csv('./top_1000_genes.csv', index_col=0)
top_1000_genes = list(top_1000_genes_temp['0'].copy())

In [5]:
X_train = X_train_temp[top_1000_genes].copy()
y_train= np.array(y_train_temp['Cluster'].copy())

In [6]:
X_train.shape

(610, 1000)

## Initial XGBoost CV

Model run on dataset with 1000 features to see how well it performs in general first. 

In [14]:
# Define the XGBoost classifier
xgb_clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

# Define k-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Evaluate accuracy
cv_scores = cross_val_score(
    xgb_clf,
    X_train, y_train,
    cv=cv,
    scoring='accuracy'
)

print("Cross-validation accuracies:", cv_scores)
print("Mean accuracy: {} ± {}".format(cv_scores.mean(), cv_scores.std()))

Cross-validation accuracies: [0.91803279 0.90163934 0.92622951 0.91803279 0.95081967]
Mean accuracy: 0.9229508196721312 ± 0.016062227821529017


## XGBoost  with Nested CV

In [17]:
# Base model
xgb_clf = xgb.XGBClassifier(
    eval_metric='logloss',
    random_state=42
)

# Parameter distributions (reasonable search space)
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': np.linspace(0.01, 0.2, 10),
    'subsample': np.linspace(0.6, 1.0, 5),
    'colsample_bytree': np.linspace(0.6, 1.0, 5),
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.2, 0.5],
}


# 2) Nested CV
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
nested_scores_auc = []
nested_scores_acc = []
inner_cv_scores = []
best_params_list = []


X = X_train.copy()
y = y_train.copy()

i=1
for train_idx, test_idx in outer_cv.split(X, y):
    X_outer_train, X_outer_test = X.iloc[train_idx], X.iloc[test_idx]
    y_outer_train, y_outer_test = y[train_idx], y[test_idx]
    
    # Inner CV for hyperparameter tuning
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    inner_search = RandomizedSearchCV(
        estimator=xgb_clf,
        param_distributions=param_dist,
        n_iter=25,                # 25 random combinations
        scoring='accuracy',        
        n_jobs=-1,
        cv=inner_cv,
        random_state=42,
        verbose=1
    )
    
    inner_search.fit(X_outer_train, y_outer_train)
    
    inner_cv_scores.append(float(inner_search.best_score_))
    best_params_list.append(inner_search.best_params_)

    
    # Evaluate best model on outer test fold
    best_inner_model = clone(inner_search.best_estimator_)
    best_inner_model.fit(X_outer_train, y_outer_train)

    y_outer_proba = best_inner_model.predict_proba(X_outer_test)[:, 1]
    auc = roc_auc_score(y_outer_test, y_outer_proba)
    nested_scores_auc.append(auc)

    acc = accuracy_score(y_outer_test, best_inner_model.predict(X_outer_test))
    nested_scores_acc.append(acc)

    print(f'Fold {i} complete')
    i=i+1

print("\nNested CV (unbiased estimate):")
print("Inner fold accuracies:", inner_cv_scores)
print("Outer fold accuracies:", nested_scores_acc)
print(f"Mean Accuracy: {np.mean(nested_scores_acc)} ± {np.std(nested_scores_acc)}")

print("Outer fold ROC-AUC:", nested_scores_auc)
print(f"Mean ROC-AUC: {np.mean(nested_scores_auc)} ± {np.std(nested_scores_auc)}")

# Summarize best parameters across folds
best_params_df = pd.DataFrame(best_params_list)
print("\nBest parameters per outer fold:")
print(best_params_df)

# Choose the most frequent or best-performing parameter set
final_best_params = best_params_df.mode().iloc[0].to_dict()
print("\nFinal chosen parameters for full training:")
print(final_best_params)


Fitting 3 folds for each of 25 candidates, totalling 75 fits
Fold 1 complete
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Fold 2 complete
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Fold 3 complete
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Fold 4 complete
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Fold 5 complete

Nested CV (unbiased estimate):
Inner fold accuracies: [0.9139463253301018, 0.9159786917114797, 0.9201191648362747, 0.9241586508116842, 0.9262162639804085]
Outer fold accuracies: [0.9344262295081968, 0.9016393442622951, 0.9262295081967213, 0.9098360655737705, 0.9016393442622951]
Mean Accuracy: 0.9147540983606559 ± 0.013318095745304868
Outer fold ROC-AUC: [0.9789819376026273, 0.9609195402298851, 0.9677002583979328, 0.9664082687338501, 0.9780361757105942]
Mean ROC-AUC: 0.970409236134978 ± 0.007000886324450185

Best parameters per outer fold:
   subsample  n_estimators  min_child_weight  max_depth  learning_rat